### Установка библиотек

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

from datasets import load_dataset
from collections import Counter
import evaluate
import random
import numpy as np

from transformers import BertConfig, BertModel, BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer, get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup

/Users/mountai_nowner/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
import gc

gc.collect()
torch.mps.empty_cache() 

In [3]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_random_seed(224)

### Загрузка датасета для анализа тональности

In [4]:
ds = load_dataset("ai-forever/kinopoisk-sentiment-classification")

### Токенизатор (и модель DeepPavlov)

In [5]:
tokenizer = BertTokenizerFast.from_pretrained('DeepPavlov/rubert-base-cased')

### Реализация срезов

In [6]:
def slice_token(index, sentences, labels, tokenizer, max_length):
    start, stop, step = index.indices(len(sentences))
    result = []
    for i in range(start, stop, step):
        encoding = tokenizer(
                sentences[i],
                padding='max_length',
                truncation = True,
                max_length = max_length,
                return_tensors = 'pt'
            )
        encoding['labels'] = [labels[i]]
        result.append({key : value[0] for key, value in encoding.items()})

    return result

### Кастомный датасет

In [7]:
class SemDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_length):
        self.sentences = sentences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):

        if isinstance(idx, slice):
            return slice_token(idx, self.sentences, self.labels, self.tokenizer, self.max_length)
        elif isinstance(idx, int):
            tokens = self.sentences[idx]
            tag = self.labels[idx]

            encoding = self.tokenizer(
                tokens,
                padding='max_length',
                truncation = True,
                max_length = self.max_length,
                return_tensors = 'pt'
            )
            encoding['labels'] = [tag]
            return {key : value[0] for key, value in encoding.items()}

### Параметры

In [ ]:
batch_size = 16
max_length = 512
epochs = 3

### Создание датасетов для тренировки, валидации и тестирования

In [9]:
dataset_train = SemDataset(ds['train']['text'], ds['train']['label'], tokenizer, max_length)
dataset_test = SemDataset(ds['test']['text'], ds['test']['label'], tokenizer, max_length)
dataset_val = SemDataset(ds['validation']['text'], ds['validation']['label'], tokenizer, max_length)

### Даталоадеры

In [10]:
test_loader = DataLoader(dataset_test, batch_size, pin_memory=True)
train_loader = DataLoader(dataset_train, batch_size)
val_loader = DataLoader(dataset_val, batch_size)

### Предобученная модель DeepPavlov/rubert-base-cased

In [ ]:
num_labels = len(set(ds['train']['label']))
model = BertForSequenceClassification.from_pretrained('DeepPavlov/rubert-base-cased', num_labels= num_labels)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Функция для подсчета метрик

In [12]:
def compute_metrics(p):

    predictions, labels = p

    predictions = predictions.argmax(axis=-1)
    cm = confusion_matrix(labels, predictions)

    print("Confusion Matrix:\n", cm)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

### Кросс-валидация

#### Аргументы на кросс-валидации

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',          
    eval_strategy="epoch",                
    per_device_train_batch_size=batch_size, 
    per_device_eval_batch_size=batch_size,  
    num_train_epochs=epochs,                      
    logging_dir="./logs",         
    logging_steps=10,             
    save_strategy="epoch",          
    load_best_model_at_end=True,  
    metric_for_best_model="f1",
    gradient_checkpointing=True
)

#### Функция для кросс-валидации

In [ ]:
def hp_space_fn(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3,log=True),
        "weight_decay" : trial.suggest_float("weight_decay", 1e-5, 0.1, log=True) 
        }

def model_init():
    return BertForSequenceClassification.from_pretrained('DeepPavlov/rubert-base-cased', num_labels=num_labels)

#### Дефолтный трейнер из transformers

In [ ]:
trainer = Trainer(
    model=model,
    model_init = model_init,
    args=training_args,
    train_dataset=train_loader.dataset,  
    eval_dataset=val_loader.dataset,  
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    )


/var/folders/6k/ps9wyw1s61s4dq5ptkr46_qc0000gn/T/ipykernel_15359/1901104278.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


#### Обучение с кросс-валидацией

In [ ]:
best_run = trainer.hyperparameter_search(
    hp_space=hp_space_fn,
    n_trials=5, 
    direction="maximize",
    backend="optuna" 
)

#### Результаты

In [ ]:
best_params = best_run.hyperparameters

### Обучение с оптимизатором и шедулером

In [12]:
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()))
total_steps = len(dataset_train) * epochs * batch_size

scheduler = get_cosine_schedule_with_warmup(
  optimizer,
  num_warmup_steps=total_steps*0.05,
  num_training_steps=total_steps
)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',          
    eval_strategy="epoch",
    learning_rate = best_params.learning_rate,
    weight_decay = best_params.weight_decay,
    per_device_train_batch_size=batch_size, 
    per_device_eval_batch_size=batch_size,  
    num_train_epochs=epochs,                      
    logging_dir="./logs",         
    logging_steps=10,             
    save_strategy="epoch",          
    load_best_model_at_end=True,  
    metric_for_best_model="f1",
    gradient_checkpointing=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader.dataset,  
    eval_dataset=val_loader.dataset,  
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler)
    )

In [ ]:
trainer.train()

eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

trainer.save_model("./sent_baseline_1_model")

### Тестирование модели

In [ ]:
def test_model(model, data_loader, device):
  model = model.eval()

  all_preds = torch.tensor([], device=device)
  all_trues = torch.tensor([], device=device)

  with torch.no_grad():
    for d in data_loader:
      input_ids = d["input_ids"].to(device)
      attention_mask = d["attention_mask"].to(device)
      targets = d["labels"].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
      )

      preds = torch.argmax(outputs['logits'], axis=-1)
      all_preds = torch.cat((all_preds, preds), -1)
      all_trues = torch.cat((all_trues, targets), -1)

  precision, recall, f1, _ = precision_recall_fscore_support(all_trues.cpu(), all_preds.cpu(), average='macro')
  acc = accuracy_score(all_trues.cpu(), all_preds.cpu())
  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall
  }

### Результаты на тестовом датасете

In [ ]:
test = test_model(model, test_loader)
test